## Imports and main

In [6]:
from pathlib import Path
import datetime
from datetime import date
import numpy as np
import pandas as pd
import csv
import re

URL = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MORTGAGE30US"
SOURCE_DIR = Path(r"C:/Users/Viv/Documents/Career readiness/internships/IDX exchange/csv")
START_YEAR, START_MONTH = 2024, 1
PROPERTY_FILTER = "Residential"
PERCENTILES = [10,25,50,75,90]
COLUMN_INDEX = {
    "CRMLSListing": 10,
    "CRMLSSold": 17
}

# Folder containing all monthly CSV files
DATA_FOLDER = Path(r"C:\Users\Viv\Documents\Career readiness\internships\IDX exchange\csv")

# Last completed calendar month
today = date.today()
last_month = today.month - 1
year = today.year
TERMINAL_FLOAT_FORMAT = lambda value: f"{value:,.6f}"

mortgage = pd.read_csv(URL, parse_dates=['observation_date'])
mortgage.columns = ['date', 'rate_30yr_fixed']

## Wk1: Properties filtered to residental only with row counts before & after

In [7]:
if last_month == 0:
    last_month = 12
    year -= 1


def valid_file(filename):
    stem = filename.stem

    # Match filenames like CRMLSSold202606 or CRMLSListing202401
    match = re.search(r"(\d{4})(\d{2})$", stem)

    if not match:
        return False

    file_year = int(match.group(1))
    file_month = int(match.group(2))

    if file_year < START_YEAR:
        return False

    if file_year > year:
        return False

    if file_year == year and file_month > last_month:
        return False

    return True


def combine_files(prefix):

    property_column = COLUMN_INDEX[prefix]

    combined = []
    header = None

    files = sorted(DATA_FOLDER.glob(f"{prefix}*.csv"))

    print(f"\n{prefix}")
    print("----------------------------")

    total_rows_before = 0

    for file in files:

        if not valid_file(file):
            continue

        with file.open("r", newline="", encoding="utf-8") as f:

            reader = csv.reader(f)

            file_header = next(reader)

            if header is None:
                header = file_header

            rows = list(reader)

            print(f"{file.name}: {len(rows)} rows")

            total_rows_before += len(rows)

            combined.extend(rows)

    # Row count after concatenation
    print(f"\nRows before concatenation: {total_rows_before}")
    print(f"Rows after concatenation:  {len(combined)}")

    before_filter = len(combined)

    residential = [
        row
        for row in combined
        if len(row) > property_column
        and row[property_column].strip() == "Residential"
    ]

    after_filter = len(residential)

    print(f"Rows before Residential filter: {before_filter}")
    print(f"Rows after Residential filter:  {after_filter}")

    output = DATA_FOLDER / f"{prefix}_Combined_Residential.csv"

    with output.open("w", newline="", encoding="utf-8") as f:

        writer = csv.writer(f)

        writer.writerow(header)

        writer.writerows(residential)

    print(f"Saved: {output.name}")



if __name__ == "__main__":
    combine_files("CRMLSListing")
    combine_files("CRMLSSold")
    



CRMLSListing
----------------------------
CRMLSListing202401.csv: 27454 rows
CRMLSListing202402.csv: 27447 rows
CRMLSListing202403.csv: 32282 rows
CRMLSListing202404.csv: 36503 rows
CRMLSListing202405.csv: 38796 rows
CRMLSListing202406.csv: 35893 rows
CRMLSListing202407.csv: 36340 rows
CRMLSListing202408.csv: 35305 rows
CRMLSListing202409.csv: 34625 rows
CRMLSListing202410.csv: 34730 rows
CRMLSListing202411.csv: 25128 rows
CRMLSListing202412.csv: 19417 rows
CRMLSListing202501.csv: 37469 rows
CRMLSListing202502.csv: 33983 rows
CRMLSListing202503.csv: 38492 rows
CRMLSListing202504.csv: 40187 rows
CRMLSListing202505.csv: 40271 rows
CRMLSListing202506.csv: 26399 rows
CRMLSListing202507.csv: 27345 rows
CRMLSListing202508.csv: 25210 rows
CRMLSListing202509.csv: 26923 rows
CRMLSListing202510.csv: 27586 rows
CRMLSListing202511.csv: 20677 rows
CRMLSListing202512.csv: 18773 rows
CRMLSListing202601.csv: 35302 rows
CRMLSListing202602.csv: 32884 rows
CRMLSListing202603.csv: 39153 rows
CRMLSListing

# WK2-3: Data from FRED merged onto sold and listings datasets using a year_month key and null count

In [8]:
from pathlib import Path
import datetime
from datetime import date
import numpy as np
import pandas as pd
import csv
import re

URL = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MORTGAGE30US"
SOURCE_DIR = Path(r"C:/Users/Viv/Documents/Career readiness/internships/IDX exchange/csv")
START_YEAR, START_MONTH = 2024, 1
PROPERTY_FILTER = "Residential"
PERCENTILES = [10,25,50,75,90]
COLUMN_INDEX = {
    "CRMLSListing": 10,
    "CRMLSSold": 17
}

# Folder containing all monthly CSV files
DATA_FOLDER = Path(r"C:\Users\Viv\Documents\Career readiness\internships\IDX exchange\csv")

# Last completed calendar month
today = date.today()
last_month = today.month - 1
year = today.year
TERMINAL_FLOAT_FORMAT = lambda value: f"{value:,.6f}"

mortgage = pd.read_csv(URL, parse_dates=['observation_date'])
mortgage.columns = ['date', 'rate_30yr_fixed']

def most_recent_completed_month():
    today = date.today()
    y, m = today.year, today.month-1
    return (y-1,12) if m==0 else (y,m)

def monthly_files(prefix):
    ey, em = most_recent_completed_month()
    return [f for p in pd.period_range(f"{START_YEAR}-{START_MONTH:02d}",f"{ey}-{em:02d}",freq="M")
            if (f:=SOURCE_DIR/f"{prefix}{p.year:04d}{p.month:02d}.csv").exists()]

def process(prefix,date_col):

    files=monthly_files(prefix)
    if not files:
        print(f"No files for {prefix}")
        return None
    frames=[pd.read_csv(f,low_memory=False) for f in files]

    print("Rows before concatenation:",sum(len(x) for x in frames))

    df=pd.concat(frames,ignore_index=True)

    print("Rows after concatenation:",len(df))

    print("Unique PropertyTypes:",sorted(df.PropertyType.dropna().astype(str).str.strip().unique()))

    print("Rows before Residential filter:",len(df))


    df=df.query("PropertyType == @PROPERTY_FILTER").copy()

    print("Rows after Residential filter:",len(df))

    nulls=pd.DataFrame({"null_count":df.isna().sum()})

    nulls["null_percent"]=nulls.null_count/len(df)*100


    print(nulls)

    print("Columns >90% null:")

    print(nulls[nulls.null_percent>90])


    stats = (
    df[["ClosePrice", "LivingArea", "DaysOnMarket"]]
    .apply(pd.to_numeric, errors="coerce")
    .describe(percentiles=[p / 100 for p in PERCENTILES])
)
    print(stats)
    
    df.to_csv(SOURCE_DIR/f"{prefix}_filtered_residential.csv",index=False)

    nulls.to_csv(SOURCE_DIR/f"{prefix}_null_summary.csv")

    stats.to_csv(SOURCE_DIR/f"{prefix}_numeric_summary.csv")

    rates=pd.read_csv(URL,parse_dates=["observation_date"])

    rates.columns=["date","rate_30yr_fixed"]

    rates["year_month"]=rates.date.dt.to_period("M")

    rates=rates.groupby("year_month",as_index=False).rate_30yr_fixed.mean()

    df["year_month"]=pd.to_datetime(df[date_col],errors="coerce").dt.to_period("M")
    # Remove rows with dates beyond the latest mortgage month
    latest_rate = rates["year_month"].max()

    before = len(df)

    df = df[df["year_month"] <= latest_rate].copy()

    print(f"Removed {before - len(df)} rows beyond available mortgage data.")
    merged=df.merge(rates,on="year_month",how="left")

    if merged["rate_30yr_fixed"].isna().any():
        missing = merged[merged["rate_30yr_fixed"].isna()]
        print(missing[[date_col, "year_month"]].head())
        raise ValueError("Mortgage-rate merge produced null values.")

    merged.to_csv(
    SOURCE_DIR / f"{prefix}_with_rates.csv",
    index=False
    )

    return merged

if __name__ == "__main__":

    sold = process("CRMLSSold", "CloseDate")
    listing = process("CRMLSListing", "ListingContractDate")

    combined = pd.concat([sold, listing], ignore_index=True, sort=False)

    combined.to_csv(
        SOURCE_DIR / "CRMLS_combined_with_rates.csv",
        index=False
    )


Rows before concatenation: 545481
Rows after concatenation: 545481
Unique PropertyTypes: ['BusinessOpportunity', 'CommercialLease', 'CommercialSale', 'Land', 'ManufacturedInPark', 'Residential', 'ResidentialIncome', 'ResidentialLease']
Rows before Residential filter: 545481
Rows after Residential filter: 367741
                              null_count  null_percent
BuyerAgentAOR                      52791     14.355484
ListAgentAOR                       46187     12.559655
Flooring                          132234     35.958460
ViewYN                             31559      8.581855
WaterfrontYN                      367506     99.936096
...                                  ...           ...
MiddleOrJuniorSchoolDistrict      367741    100.000000
OriginatingSystemName             294467     80.074563
OriginatingSystemSubName          294467     80.074563
BuyerAgencyCompensationType       321605     87.454214
BuyerAgencyCompensation           321616     87.457205

[82 rows x 2 columns]
Colu

# WK4-5: Data Cleaning

In [ ]:
from pathlib import Path
import datetime
from datetime import date
import numpy as np
import pandas as pd
import csv
import re

URL = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MORTGAGE30US"
SOURCE_DIR = Path(r"C:/Users/Viv/Documents/Career readiness/internships/IDX exchange/csv")
START_YEAR, START_MONTH = 2024, 1
PROPERTY_FILTER = "Residential"
PERCENTILES = [10,25,50,75,90]
COLUMN_INDEX = {
    "CRMLSListing": 10,
    "CRMLSSold": 17
}

# Folder containing all monthly CSV files
DATA_FOLDER = Path(r"C:\Users\Viv\Documents\Career readiness\internships\IDX exchange\csv")

# Last completed calendar month
today = date.today()
last_month = today.month - 1
year = today.year
TERMINAL_FLOAT_FORMAT = lambda value: f"{value:,.6f}"

mortgage = pd.read_csv(URL, parse_dates=['observation_date'])
mortgage.columns = ['date', 'rate_30yr_fixed']

def most_recent_completed_month():
    today = date.today()
    y, m = today.year, today.month-1
    return (y-1,12) if m==0 else (y,m)

def monthly_files(prefix):
    ey, em = most_recent_completed_month()
    return [f for p in pd.period_range(f"{START_YEAR}-{START_MONTH:02d}",f"{ey}-{em:02d}",freq="M")
            if (f:=SOURCE_DIR/f"{prefix}{p.year:04d}{p.month:02d}.csv").exists()]

def process(prefix,date_col):

    files=monthly_files(prefix)
    if not files:
        print(f"No files for {prefix}")
        return None
    frames=[pd.read_csv(f,low_memory=False) for f in files]

    print("Rows before concatenation:",sum(len(x) for x in frames))

    df=pd.concat(frames,ignore_index=True)

    print("Rows after concatenation:",len(df))

    print("Unique PropertyTypes:",sorted(df.PropertyType.dropna().astype(str).str.strip().unique()))

    print("Rows before Residential filter:",len(df))


    df=df.query("PropertyType == @PROPERTY_FILTER").copy()

    print("Rows after Residential filter:",len(df))

    nulls=pd.DataFrame({"null_count":df.isna().sum()})

    nulls["null_percent"]=nulls.null_count/len(df)*100


    print(nulls)

    print("Columns >90% null:")

    print(nulls[nulls.null_percent>90])


    stats = (
    df[["ClosePrice", "LivingArea", "DaysOnMarket"]]
    .apply(pd.to_numeric, errors="coerce")
    .describe(percentiles=[p / 100 for p in PERCENTILES])
)
    print(stats)
    
    df.to_csv(SOURCE_DIR/f"{prefix}_filtered_residential.csv",index=False)

    nulls.to_csv(SOURCE_DIR/f"{prefix}_null_summary.csv")

    stats.to_csv(SOURCE_DIR/f"{prefix}_numeric_summary.csv")

    rates=pd.read_csv(URL,parse_dates=["observation_date"])

    rates.columns=["date","rate_30yr_fixed"]

    rates["year_month"]=rates.date.dt.to_period("M")

    rates=rates.groupby("year_month",as_index=False).rate_30yr_fixed.mean()

    df["year_month"]=pd.to_datetime(df[date_col],errors="coerce").dt.to_period("M")
    # Remove rows with dates beyond the latest mortgage month
    latest_rate = rates["year_month"].max()

    before = len(df)

    df = df[df["year_month"] <= latest_rate].copy()

    print(f"Removed {before - len(df)} rows beyond available mortgage data.")
    merged=df.merge(rates,on="year_month",how="left")

    if merged["rate_30yr_fixed"].isna().any():
        missing = merged[merged["rate_30yr_fixed"].isna()]
        print(missing[[date_col, "year_month"]].head())
        raise ValueError("Mortgage-rate merge produced null values.")

    merged.to_csv(
    SOURCE_DIR / f"{prefix}_with_rates.csv",
    index=False
    )

    return merged

if __name__ == "__main__":

    sold = process("CRMLSSold", "CloseDate")
    listing = process("CRMLSListing", "ListingContractDate")

    combined = pd.concat([sold, listing], ignore_index=True, sort=False)

    combined.to_csv(
        SOURCE_DIR / "CRMLS_combined_with_rates.csv",
        index=False
    )



# ===========================
    # WEEK 4-5 REPORTING
    # ===========================

def print_transformation_log():
    print("\n===== Transformation Log =====")

    steps = [
        ("Concatenate monthly files",
         "Creates one dataset for each MLS feed."),
        ("Filter Residential records",
         "Removes non-residential property types."),
        ("Null summary",
         "Measures missing values by column."),
        ("Numeric summary",
         "Summarizes important numeric fields."),
        ("Convert dates",
         "Creates year_month merge key."),
        ("Merge mortgage rates",
         "Adds monthly average 30-year mortgage rate from FRED."),
        ("Remove future dates",
         "Ensures every record has a matching mortgage rate."),
    ]

    for i, (step, reason) in enumerate(steps, start=1):
        print(f"{i}. {step}: {reason}")


def print_row_count_summary(before_rows, after_rows):

    print("\n===== Row Count Summary =====")
    print(f"Rows before processing : {before_rows:,}")
    print(f"Rows after processing  : {after_rows:,}")
    print(f"Rows removed           : {before_rows-after_rows:,}")




def save_dtype_summary(df, output_dir):

    dtypes = pd.DataFrame({
        "Column": df.columns,
        "DataType": df.dtypes.astype(str)
    })

    print("\n===== Data Types =====")
    print(dtypes)

    dtypes.to_csv(
        output_dir / "data_type_summary.csv",
        index=False
    )



def date_consistency_summary(df):

    print("\n===== Date Consistency =====")

    date_columns = [
        c for c in df.columns
        if "Date" in c
    ]

    summary = []

    for col in date_columns:

        converted = pd.to_datetime(
            df[col],
            errors="coerce"
        )

        invalid = converted.isna().sum()

        summary.append({
            "Column": col,
            "Invalid Dates": invalid
        })

    summary = pd.DataFrame(summary)

    print(summary)

    summary.to_csv(
        SOURCE_DIR / "date_consistency_summary.csv",
        index=False
    )



def geographic_summary(df):

    print("\n===== Geographic Quality =====")

    if (
        "Latitude" not in df.columns or
        "Longitude" not in df.columns
    ):
        print("Latitude/Longitude columns not found.")
        return

    invalid = df[
        (~df["Latitude"].between(-90,90)) |
        (~df["Longitude"].between(-180,180))
    ]

    print(f"Invalid coordinate records: {len(invalid)}")

    invalid.to_csv(
        SOURCE_DIR / "invalid_coordinates.csv",
        index=False
    )





    print_transformation_log()

    # Row count summary
    before_rows = 0

    for prefix in ["CRMLSSold", "CRMLSListing"]:
        files = monthly_files(prefix)
        for f in files:
            before_rows += len(pd.read_csv(f, low_memory=False))

    after_rows = len(combined)

    print_row_count_summary(before_rows, after_rows)

    # Save data types
    save_dtype_summary(combined, SOURCE_DIR)

    # Date consistency
    date_consistency_summary(combined)

    # Geographic quality
    geographic_summary(combined)

    # Numeric validation
    validate_numeric_fields(combined, SOURCE_DIR, "combined")

    print("\nWeek 4-5 reporting complete.")


def validate_numeric_fields(df, output_dir, prefix):

    checks = []

    # ClosePrice
    if "ClosePrice" in df.columns:
        price = pd.to_numeric(df["ClosePrice"], errors="coerce")

        checks.append({
            "Field": "ClosePrice",
            "Negative Values": (price < 0).sum(),
            "Zero Values": (price == 0).sum(),
            "Null Values": price.isna().sum()
        })

    # LivingArea
    if "LivingArea" in df.columns:
        area = pd.to_numeric(df["LivingArea"], errors="coerce")

        checks.append({
            "Field": "LivingArea",
            "Negative Values": (area < 0).sum(),
            "Zero Values": (area == 0).sum(),
            "Null Values": area.isna().sum()
        })

    # DaysOnMarket
    if "DaysOnMarket" in df.columns:
        dom = pd.to_numeric(df["DaysOnMarket"], errors="coerce")

        checks.append({
            "Field": "DaysOnMarket",
            "Negative Values": (dom < 0).sum(),
            "Zero Values": (dom == 0).sum(),
            "Null Values": dom.isna().sum()
        })

    validation = pd.DataFrame(checks)

    print("\n===== Numeric Validation =====")
    print(validation)

    validation.to_csv(
        output_dir / f"{prefix}_numeric_validation.csv",
        index=False
    )

    return validation

Rows before concatenation: 545481
Rows after concatenation: 545481
Unique PropertyTypes: ['BusinessOpportunity', 'CommercialLease', 'CommercialSale', 'Land', 'ManufacturedInPark', 'Residential', 'ResidentialIncome', 'ResidentialLease']
Rows before Residential filter: 545481
Rows after Residential filter: 367741
                              null_count  null_percent
BuyerAgentAOR                      52791     14.355484
ListAgentAOR                       46187     12.559655
Flooring                          132234     35.958460
ViewYN                             31559      8.581855
WaterfrontYN                      367506     99.936096
...                                  ...           ...
MiddleOrJuniorSchoolDistrict      367741    100.000000
OriginatingSystemName             294467     80.074563
OriginatingSystemSubName          294467     80.074563
BuyerAgencyCompensationType       321605     87.454214
BuyerAgencyCompensation           321616     87.457205

[82 rows x 2 columns]
Colu

# WK 6 

In [11]:
# Submit a .py script demonstrating all engineered metrics (price ratio, close-to-original-list ratio, PPSF, days
# on market, YrMo, listing-to-contract days, contract-to-close days), with a sample output table showing the
# new columns populated correctly. Include at least one segmented summary table grouped by
# PropertyType or CountyOrParish.

# Price Ratio ClosePrice / OriginalListPrice Measures negotiation strength
# Price Per Sq Ft ClosePrice / LivingArea Normalizes price across sizes
# Days on Market DaysOnMarket (raw field) Time-to-sell indicator
# Year / Month / YrMo Derived from CloseDate Enables time-series analysis
# Close to Original List
# Ratio
# ClosePrice / OriginalListPrice Captures full price reduction history
# Listing to Contract Days PurchaseContractDate -
# ListingContractDate
# Measures time from listing to accepted
# offer
# Contract to Close Days CloseDate - PurchaseContractDate Escrow and closing period duration
# Add school districts using the properties’ latitude and longitude values
# https://data.ca.gov/dataset/california-school-district-areas-2024-25/resource/7dfaf005-58eb-45db-93b1-7aff091b2172